# 1. Importing libraries

In [44]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels as sm
import scipy.stats as stats

# Time-series econometrics
from statsmodels.tsa.api import VAR


# 2. Preparing the data

In [45]:
df = pd.read_csv("../data/processed/weekly_cleaned_data.csv", index_col="Dates", parse_dates=True)
print(df.sample(4))

             ret_wti  er_sp500  d_spread_hy    rv_wti  rv_sp500         rv_hy
Dates                                                                        
2001-09-14  0.053526  0.005655     0.006590  0.003890  0.000039  2.986230e-05
2025-11-07 -0.022555 -0.017105     0.001320  0.000460  0.000284  2.628952e-06
1998-10-16 -0.028210  0.069876     0.005190  0.000473  0.002048  2.256530e-05
2008-08-15 -0.012732  0.000992     0.000076  0.001005  0.000250  8.282260e-07


# 3. Testing for Stationarity

In [46]:
variables = df.columns.tolist()
for i in variables:
    result = sm.tsa.stattools.adfuller(df[i])
    print(f"{i}: ADF = {result[0]:.4f}, nb of lags used {result[2]}, p-value = {result[1]:.6f}")

ret_wti: ADF = -11.8303, nb of lags used 8, p-value = 0.000000
er_sp500: ADF = -20.5501, nb of lags used 3, p-value = 0.000000
d_spread_hy: ADF = -7.9360, nb of lags used 22, p-value = 0.000000
rv_wti: ADF = -7.4003, nb of lags used 10, p-value = 0.000000
rv_sp500: ADF = -8.5132, nb of lags used 6, p-value = 0.000000
rv_hy: ADF = -6.3951, nb of lags used 11, p-value = 0.000000


# 4. Number of lags selection

In [47]:
# VAR 1 lag selection (returns)
var1_data = df[["ret_wti", "er_sp500", "d_spread_hy"]]
model1 = VAR(var1_data)
results1 = model1.select_order(maxlags=20)
print(results1.summary())
print(select_lag_lr(var1_data))

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -25.49      -25.48   8.527e-12      -25.48
1       -25.64     -25.59*   7.330e-12      -25.62
2       -25.66      -25.58   7.198e-12     -25.63*
3       -25.66      -25.55   7.197e-12      -25.62
4       -25.66      -25.52   7.149e-12      -25.61
5       -25.67      -25.49   7.103e-12      -25.60
6       -25.67      -25.46   7.108e-12      -25.59
7       -25.68      -25.43   7.064e-12      -25.58
8       -25.68      -25.40   7.032e-12      -25.58
9       -25.68      -25.37   7.030e-12      -25.56
10      -25.68      -25.33   7.058e-12      -25.55
11     -25.68*      -25.30  7.020e-12*      -25.54
12      -25.68      -25.27   7.062e-12      -25.52
13      -25.67      -25.23   7.073e-12      -25.51
14      -25.67      -25.20   7.086e-12      -25.49
15      -25.67      -25.16   7.134e-12      -25.48
16      -25.66      -25.11   7.

c:\Users\larao\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-FRI will be used.
  self._init_dates(dates, freq)
c:\Users\larao\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-FRI will be used.
  self._init_dates(dates, freq)
c:\Users\larao\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-FRI will be used.
  self._init_dates(dates, freq)
c:\Users\larao\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-FRI will be used.
  self._init_dates(dates, freq)
c:\Users\larao\AppData\Local\Programs\Python

interpretation de des resultats de la selection 2

In [48]:
# VAR 2 lag selection (realized variances)
var2_data = df[["rv_wti", "rv_hy", "rv_sp500"]]
model2 = VAR(var2_data)
var2_lag_results = model2.select_order(maxlags=20)
print(var2_lag_results.summary())

c:\Users\larao\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-FRI will be used.
  self._init_dates(dates, freq)


 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -48.79      -48.78   6.437e-22      -48.79
1       -49.78      -49.74   2.404e-22      -49.76
2       -49.89     -49.81*   2.152e-22      -49.86
3       -49.91      -49.79   2.120e-22      -49.86
4       -49.92      -49.78   2.085e-22     -49.87*
5       -49.92      -49.74   2.088e-22      -49.85
6       -49.92      -49.71   2.085e-22      -49.84
7       -49.93      -49.68   2.072e-22      -49.84
8       -49.93      -49.65   2.075e-22      -49.82
9       -49.92      -49.61   2.082e-22      -49.81
10      -49.93      -49.58   2.077e-22      -49.80
11      -49.93      -49.55   2.075e-22      -49.79
12      -49.93      -49.52   2.065e-22      -49.78
13     -49.93*      -49.49  2.060e-22*      -49.77
14      -49.93      -49.45   2.070e-22      -49.75
15      -49.93      -49.42   2.071e-22      -49.74
16      -49.92      -49.38   2.

# Model VAR 2 (variances)

In [49]:
#fitting the model
var2_fitted_results = model2.fit(4)
print(var2_fitted_results.summary())

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Fri, 27, Mar, 2026
Time:                     17:42:50
--------------------------------------------------------------------
No. of Equations:         3.00000    BIC:                   -49.7474
Nobs:                     1436.00    HQIC:                  -49.8371
Log likelihood:           29747.6    FPE:                2.15185e-22
AIC:                     -49.8905    Det(Omega_mle):     2.09445e-22
--------------------------------------------------------------------
Results for equation rv_wti
                 coefficient       std. error           t-stat            prob
------------------------------------------------------------------------------
const               0.000610         0.000097            6.298           0.000
L1.rv_wti           0.270146         0.027358            9.875           0.000
L1.rv_hy           36.369729         7.895391            4.606     

## Cholesky decomposition

In [50]:
# extracting the residual covariance matrix
var2_cov_matrix = var2_fitted_results.sigma_u.to_numpy()

#Decompose it
P = np.linalg.cholesky(var2_cov_matrix)

#obtaining structural impusle vector 
shock_vector = np.array ([1,0,0])
impulse_vector = P @ shock_vector
print(impulse_vector)

[2.52727627e-03 1.98358656e-06 1.57063833e-04]


# getting irfs by code

In [54]:
# extract pure coeff matrices
A_matrices = var2_fitted_results.coefs
print(A_matrices)

full_irf_system = []

#for h in range(12):



[[[ 2.70145995e-01  3.63697294e+01  3.48924768e-01]
  [ 2.70511628e-04  2.49393660e-01  1.40049087e-03]
  [ 1.77310717e-02  6.33505161e+00  4.35047382e-01]]

 [[ 1.29095371e-01 -1.28212117e+01 -7.20956107e-03]
  [ 9.71895869e-05  8.39334573e-02  7.32448934e-04]
  [-1.78634899e-02  6.21535293e+00  1.83382078e-01]]

 [[ 6.32416806e-02  1.53771761e+01 -5.02119968e-02]
  [-1.94113913e-04 -7.68870043e-03  6.44810401e-04]
  [-4.33971202e-03  1.06791127e+00  8.84350480e-02]]

 [[ 1.28500837e-01 -1.33536867e+01  2.01537074e-01]
  [ 3.88883542e-04 -2.80154194e-02  3.62124411e-05]
  [ 3.63560170e-03 -7.72435042e-01  1.11396663e-02]]]


In [52]:
# 1. Your manual calculation (the structural shock to WTI)
print("My Manual Vector:", impulse_vector)

# 2. Statsmodels' automatic calculation of the t=0 shock to WTI
irfs = var2_fitted_results.irf(periods = 12)
print("Statsmodels IRF Vector:", irfs.orth_irfs[0, :, 0])


My Manual Vector: [2.52727627e-03 1.98358656e-06 1.57063833e-04]
Statsmodels IRF Vector: [2.52727627e-03 1.98358656e-06 1.57063833e-04]
